In [1]:
def read_file_and_generate_structure(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    cantos = []
    current_canto = []
    current_paragraph = []

    for line in lines:
        stripped_line = line.strip()
        if stripped_line.startswith("Canto"):
            if current_paragraph:
                current_canto.append("\n".join(current_paragraph))
                current_paragraph = []
            if current_canto:
                cantos.append(current_canto)
                current_canto = []
        elif stripped_line == "":
            if current_paragraph:
                current_canto.append("\n".join(current_paragraph))
                current_paragraph = []
        else:
            current_paragraph.append(stripped_line)

    if current_paragraph:
        current_canto.append("\n".join(current_paragraph))
    if current_canto:
        cantos.append(current_canto)

    return cantos

# Path to the file
file_path = r"48-77gptout.txt"

# Generate the structure
cantos_structure = read_file_and_generate_structure(file_path)

In [2]:
def read_file_and_generate_summaries(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    summaries = []
    current_summary = []

    for line in lines:
        stripped_line = line.strip()
        if stripped_line.startswith("Canto"):
            if current_summary:
                summaries.append("\n".join(current_summary))
                current_summary = []
        else:
            current_summary.append(stripped_line)

    if current_summary:
        summaries.append("\n".join(current_summary))

    return summaries

# Path to the file
file_path = r"48-77summaries.txt"

# Generate the summaries
summaries = read_file_and_generate_summaries(file_path)

In [3]:
from llama_index.core import Document
from llama_index.core.schema import MetadataMode

documents = []
para_count = 0
for i in range(len(cantos_structure)):
    for paragraph in cantos_structure[i]:        
        document = Document(
        text = paragraph,
        metadata={
            "canto_summary": summaries[i],
            "r_id": para_count
        },
        excluded_llm_metadata_keys=["r_id"],
        metadata_seperator="::",
        metadata_template="{value}",
        text_template="Content: {content}, Optional Chapter summary: {metadata_str}\n",
        )
        documents.append(document)
        para_count += 1

In [7]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=512, context_window = 2048, num_output=256))

index = VectorStoreIndex.from_documents(documents)
index.storage_context.persist(persist_dir="balakanda_recursive_test")

c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_15648\2948058279.py:13: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=512, context_window = 2048, num_output=256))
